In [1]:
pip install torch torchvision opencv-python matplotlib numpy

  Using cached numpy-2.2.6-cp311-cp311-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.19.0 which is incompatible.

[notice] A new release of pip is available: 23.1.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        # grad_output[0] contains the gradients for the feature map
        self.gradients = grad_output[0]

    def __call__(self, input_tensor, target_class=None):
        # 1. Forward pass
        self.model.eval()
        output = self.model(input_tensor)
        
        # If no target class is specified, use the model's top prediction
        if target_class is None:
            target_class = torch.argmax(output, dim=1).item()
            
        # 2. Zero gradients and do a backward pass for the target class
        self.model.zero_grad()
        target_score = output[0, target_class]
        target_score.backward(retain_graph=True)
        
        # 3. Calculate the Grad-CAM heatmap
        # Get the gradients and activations
        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]
        
        # Global average pooling on the gradients to get the weights
        weights = np.mean(gradients, axis=(1, 2))
        
        # Multiply each activation channel by its weight
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i, :, :]
            
        # Apply ReLU to only keep features that have a positive influence on the class
        cam = np.maximum(cam, 0)
        
        # Normalize the heatmap between 0 and 1
        cam = cam - np.min(cam)
        cam = cam / np.max(cam)
        
        return cam

In [3]:
def show_cam_on_image(img_path, mask, alpha=0.5):
    # Load original image
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.float32(img) / 255
    
    # Resize mask to match the image dimensions
    heatmap = cv2.resize(mask, (img.shape[1], img.shape[0]))
    
    # Apply color map (JET is standard for Grad-CAM)
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255
    
    # Combine image and heatmap
    cam_img = heatmap * alpha + img * (1 - alpha)
    cam_img = cam_img / np.max(cam_img)
    
    # Plotting
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(img)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM")
    plt.imshow(cam_img)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

In [4]:
# 1. Setup device and load pre-trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained=True).to(device)

# 2. Define the target layer 
# For ResNet, the last convolutional layer is inside layer4
target_layer = model.layer4[-1].conv3

# 3. Initialize Grad-CAM
grad_cam = GradCAM(model, target_layer)

# 4. Prepare the image
image_path = "your_image_here.jpg"  # Replace with your image path
img_pil = Image.open(image_path).convert('RGB')

# Standard ImageNet transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

input_tensor = transform(img_pil).unsqueeze(0).to(device)

# 5. Generate the heatmap
# If you don't pass a target_class, it defaults to the model's highest prediction
heatmap = grad_cam(input_tensor)

# 6. Visualize
show_cam_on_image(image_path, heatmap)

c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\Asus/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


 49%|████▉     | 47.9M/97.8M [00:27<00:28, 1.82MB/s]


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

import torch
import numpy as np
import cv2
class ViTSegmentationGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        
        self.hook_handles.append(target_layer.register_forward_hook(self.save_activation))
        self.hook_handles.append(target_layer.register_backward_hook(self.save_gradient))

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def process_vit_tokens(self, tensor):
        tokens = tensor[:, 1:, :]
        b, n, c = tokens.shape
        h = w = int(n ** 0.5)
        spatial_map = tokens.transpose(1, 2).reshape(b, c, h, w)
        return spatial_map

    def generate_heatmap(self, input_tensor, target_class):
        self.model.eval()
        self.model.zero_grad()
        logits = self.model(input_tensor)
        pred_mask = logits.argmax(dim=1)
        class_mask = (pred_mask == target_class).float()
        
        if class_mask.sum() == 0:
            score = logits[0, target_class, :, :].sum()
        else:
            score = (logits[0, target_class, :, :] * class_mask[0]).sum()
        score.backward(retain_graph=True)
        
        acts = self.process_vit_tokens(self.activations)[0].cpu().data.numpy()
        grads = self.process_vit_tokens(self.gradients)[0].cpu().data.numpy()
        
        weights = np.mean(grads, axis=(1, 2))
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * acts[i, :, :]
            
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))
        cam = cam - np.min(cam)
        cam_max = np.max(cam)
        if cam_max != 0:
            cam = cam / cam_max
            
        return cam

    def remove_hooks(self):
        """Removes PyTorch hooks to prevent memory leaks."""
        for handle in self.hook_handles:
            handle.remove()

def show_bitemporal_cam(input_tensor, heatmap):
    post_img = input_tensor[0, 3:6, :, :].cpu().numpy().transpose(1, 2, 0)
    post_img = (post_img - post_img.min()) / (post_img.max() - post_img.min() + 1e-8)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = np.float32(heatmap_colored) / 255
    heatmap_colored = heatmap_colored[:, :, ::-1] 
    overlay = 0.5 * heatmap_colored + 0.5 * post_img
    overlay = np.clip(overlay, 0, 1)
    
    return post_img, heatmap_colored, overlay